In [ ]:
# Surface source + Floquet periodic BCs

from netgen.occ import *
from netgen.meshing import IdentificationType
from ngsolve import *
from ngsolve.webgui import Draw

# 1. Geometry Setup
L = 2.0
Lx = L/8
Ly = L/128

# Upper Air (Between Source and Rigid Wall)
geometry = Box((0, 0, 0), (Lx, Ly, L))
geometry.mat("air")     # <-- Added material name

# Tag the boundaries for Boundary Conditions
geometry.faces.Max(Z).name = "top"
geometry.faces.Min(Z).name = "bottom"

# 3. Identify Periodic Faces (on the outer boundaries of the glued geometry)
geometry.faces.Max(X).Identify(geometry.faces.Min(X), "periodic_x")
geometry.faces.Max(Y).Identify(geometry.faces.Min(Y), "periodic_y")

# 4. Generate Mesh
geo = OCCGeometry(geometry)
mesh = Mesh(geo.GenerateMesh(maxh=0.05))

print("Mesh generated successfully! Notice the internal face at z=0.2")
Draw(mesh)

Mesh generated successfully! Notice the internal face at z=0.2


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [59]:
import cmath, math
# 1. Physics Parameters
freq = 343.0     
c_0 = 343.0* (1 - 1j * 0.001)  # Complex speed of sound to introduce a small amount of damping
k = 2 * math.pi * freq / c_0  

# Incident angles (e.g., 45 degrees)
theta = (math.pi / 4)  # Polar angle (0 for normal incidence)
phi = 0.0

# Wave vector components
kx = k * math.sin(theta) * math.cos(phi)
ky = k * math.sin(theta) * math.sin(phi)
kz = k * math.cos(theta)

# Transverse wave vector for the Floquet shift
k_vec = CF((kx, ky, 0)) 

# 2. Finite Element Space
fes = Periodic(H1(mesh, order=3, complex=True))

u, v = fes.TnT()

print(f"Degrees of freedom: {fes.ndof}")
print(f"wave vector k: ({kx:.2f}, {ky:.2f}, {kz:.2f})")


Degrees of freedom: 9891
wave vector k: (4.44+0.00j, 0.00+0.00j, 4.44+0.00j)


In [60]:
# FEM model

# 2. Modified Gradients (Floquet Trick)
grad_u = grad(u) + 1j * k_vec * u
grad_v = grad(v) - 1j * k_vec * v 

# 3. Bilinear Form (LHS)
Z_fem = BilinearForm(fes)
Z_fem += (grad_u * grad_v - k**2 * u * v) * dx

# 4. Linear Form (RHS) - The Transparent Source
s_fem = LinearForm(fes)
source_func = 1.0  
s_fem += source_func* v * ds("bottom")

# 5. Assemble and Solve
gfu = GridFunction(fes)

with TaskManager():
    Z_fem.Assemble()
    s_fem.Assemble()
    gfu.vec.data = Z_fem.mat.Inverse(fes.FreeDofs()) * s_fem.vec
    
print("Solve complete!")


Solve complete!


In [62]:
# 1. Reconstruct Field
phase = exp(1j * (kx * x + ky * y))
p_true = gfu * phase

# 3. Visualize
print("Rendering total pressure field (PML zeroed out)...")
Draw(p_true, mesh, "Acoustic Pressure", animate_complex=True)

Rendering total pressure field (PML zeroed out)...


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 's…

BaseWebGuiScene